### Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading


In [ ]:
# Load the dataset
df = pd.read_csv('lung_cancer.csv')

# Display the first 5 rows
df.head()

In [ ]:
# Check the size of the dataset
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
# Display column names and data types
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

## 2. Exploratory Data Analysis


In [ ]:
# Class distribution
class_counts = df['lung_cancer_risk'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(['Low Risk (0)', 'High Risk (1)'], class_counts.values, color=['steelblue', 'tomato'])
plt.title('Class Distribution: Lung Cancer Risk')
plt.xlabel('Class')
plt.ylabel('Number of Patients')
plt.tight_layout()
plt.show()

print(f"Low Risk  (0): {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"High Risk (1): {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")

In [ ]:
# Compare key continuous features between risk groups
continuous_features = ['pack_years', 'crp_level', 'age', 'oxygen_saturation', 'fev1_x10', 'air_pollution_index']

group_means = df.groupby('lung_cancer_risk')[continuous_features].mean().round(2)
group_means.index = ['Low Risk (0)', 'High Risk (1)']
print("Mean values of continuous features by risk group:")
group_means

In [ ]:
# Compare key binary features between risk groups
binary_features = ['smoker', 'xray_abnormal', 'chronic_cough', 'copd', 'family_history_cancer', 'passive_smoking']

binary_means = df.groupby('lung_cancer_risk')[binary_features].mean().round(3)
binary_means.index = ['Low Risk (0)', 'High Risk (1)']
print("Proportion of binary features by risk group:")
binary_means

## 3. Feature Selection

We compute the Pearson correlation between each feature and the target variable (`lung_cancer_risk`).
Features with a very low absolute correlation (below 0.05) contribute little signal and can be removed.


In [ ]:
# Compute correlation of each feature with the target
correlations = df.corr()['lung_cancer_risk'].drop('lung_cancer_risk').abs().sort_values(ascending=False)

# Plot
plt.figure(figsize=(10, 7))
correlations.plot(kind='barh', color='steelblue', edgecolor='black')
plt.axvline(x=0.05, color='red', linestyle='--', label='Threshold (0.05)')
plt.title('Feature Correlation with Lung Cancer Risk')
plt.xlabel('Absolute Pearson Correlation')
plt.legend()
plt.tight_layout()
plt.show()

print("\nCorrelation values:")
print(correlations.to_string())

In [ ]:
# Keep only features above the threshold
threshold = 0.05
selected_features = correlations[correlations >= threshold].index.tolist()

print(f"Features selected ({len(selected_features)}):")
for f in selected_features:
    print(f"  - {f}")

## 4. Data Preprocessing

Steps:
1. Separate features (X) and target (y)
2. Apply **StandardScaler** to continuous features
3. Split into training (80%) and testing (20%) sets using stratification


In [ ]:
# Separate features and target
X = df[selected_features]
y = df['lung_cancer_risk']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Define which selected features are continuous (need scaling)
continuous_cols = ['age', 'education_years', 'smoking_years', 'cigarettes_per_day', 'pack_years',
                   'air_pollution_index', 'bmi', 'oxygen_saturation', 'fev1_x10', 'crp_level',
                   'exercise_hours_per_week', 'alcohol_units_per_week']

# Only scale the continuous columns that are present in selected features
cols_to_scale = [c for c in continuous_cols if c in X.columns]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

print("Scaling applied to:", cols_to_scale)
X_scaled.head()

In [ ]:
# Split into training (80%) and testing (20%) sets
# stratify=y ensures both sets have the same class ratio as the original dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape}")
print(f"Testing set size  : {X_test.shape}")

print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

## 5. K-Nearest Neighbour (KNN) Classifier


#### Step 1: Train with k=5 and evaluate:

In [ ]:
# Train KNN with k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict on the test set
y_pred_knn = knn.predict(X_test)

# Evaluate
print("--- KNN (k=5) Performance ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_knn):.4f}")

#### Step 2: Find the optimal value of k:

In [ ]:
# Test different k values to find the best one
k_values = range(1, 21, 2)
train_accuracy = []
test_accuracy = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, y_train)
    train_accuracy.append(knn_k.score(X_train, y_train))
    test_accuracy.append(knn_k.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_accuracy, marker='o', label='Training Accuracy')
plt.plot(k_values, test_accuracy, marker='o', label='Test Accuracy')
plt.title('KNN Accuracy vs k Value')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### Step 3: Retrain with optimal k and final evaluation:

In [ ]:
# Find the best k
optimal_k = list(k_values)[test_accuracy.index(max(test_accuracy))]
print(f"Optimal k: {optimal_k}")

# Retrain with optimal k
knn_best = KNeighborsClassifier(n_neighbors=optimal_k)
knn_best.fit(X_train, y_train)
y_pred_knn_best = knn_best.predict(X_test)

# Final metrics
print(f"\n--- KNN (k={optimal_k}) Final Performance ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_knn_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn_best):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_knn_best):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_knn_best):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn_best, target_names=['Low Risk', 'High Risk']))

In [ ]:
# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=['Low Risk', 'High Risk'])
disp.plot(cmap='Blues')
plt.title(f'KNN Confusion Matrix (k={optimal_k})')
plt.tight_layout()
plt.show()

## 6. Artificial Neural Network (ANN) Classifier


#### Step 1: Build the model:

In [ ]:
# Build the ANN model
model = Sequential()

# Input layer (one neuron per feature)
model.add(Input(shape=(X_train.shape[1],)))

# First hidden layer: 16 neurons, ReLU activation
model.add(Dense(16, activation='relu'))

# Second hidden layer: 8 neurons, ReLU activation
model.add(Dense(8, activation='relu'))

# Output layer: 1 neuron, sigmoid (binary classification)
model.add(Dense(1, activation='sigmoid'))

# Display model structure
model.summary()

#### Step 2: Compile the model:

In [ ]:
# Compile with Adam optimizer and binary cross-entropy loss
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#### Step 3: Train the model:

In [ ]:
# Train the model for 30 epochs
history = model.fit(
    X_train, y_train,
    epochs=30,
    validation_split=0.2,
    verbose=1
)

#### Step 4: Plot the training history:

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

#### Step 5: Evaluate the model:

In [ ]:
# Evaluate on the test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss    : {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get predictions (convert probabilities to 0/1)
y_pred_prob = model.predict(X_test, verbose=0)
y_pred_ann = (y_pred_prob >= 0.5).astype(int).flatten()

# Full metrics
print("\n--- ANN Final Performance ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_ann):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ann):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_ann):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_ann):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ann, target_names=['Low Risk', 'High Risk']))

In [ ]:
# Confusion Matrix
cm_ann = confusion_matrix(y_test, y_pred_ann)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_ann, display_labels=['Low Risk', 'High Risk'])
disp.plot(cmap='Oranges')
plt.title('ANN Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Model Comparison


In [ ]:
# Summary comparison table
results = {
    'Model': [f'KNN (k={optimal_k})', 'ANN'],
    'Accuracy' : [accuracy_score(y_test, y_pred_knn_best), accuracy_score(y_test, y_pred_ann)],
    'Precision': [precision_score(y_test, y_pred_knn_best), precision_score(y_test, y_pred_ann)],
    'Recall'   : [recall_score(y_test, y_pred_knn_best), recall_score(y_test, y_pred_ann)],
    'F1-Score' : [f1_score(y_test, y_pred_knn_best), f1_score(y_test, y_pred_ann)],
}

results_df = pd.DataFrame(results).set_index('Model').round(4)
print(results_df.to_string())

In [ ]:
# Bar chart comparison
results_df.plot(kind='bar', figsize=(9, 5), color=['steelblue', 'tomato'])
plt.title('KNN vs ANN - Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()